# Crawlspace Rot: Batch Cluster Writer

Discovers all `service_page_cluster_*.md` files in `crawlspace-rot/src/data/generated_content/`,
generates content for every pending subtopic section in each file, and writes results back.

- Files with `status: draft` are skipped automatically
- Each subtopic receives the **full document** + **Top-K RAG context**
- References are renumbered globally per file and written to a `## References` table
- Set `DRY_RUN = True` to preview discovered files without writing

In [1]:
from pathlib import Path

LANCEDB_PATH = r"C:\Users\tfalcon\microsites\tools\programatic writing stuffs\DBA writer\lancedb_construction_books"
LANCEDB_TABLE = "construction_books"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
MODEL = "gpt-4o"
TEMPERATURE = 0.7
TOP_K = 20
WORD_TARGET = 200

DRY_RUN = False  # Set True to preview without writing

CRAWLSPACE_ROT_DIR = Path("../../../apps/crawlspace-rot/src/data/generated_content")
AGENTS_DIR = Path("../agents")
CONTENT_REVIEW_DIR = Path("../../content-review")

In [2]:
import sys, os, re
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.insert(0, str(CONTENT_REVIEW_DIR.resolve()))

import lancedb
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from agent_loader import load_agent

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
embedder = SentenceTransformer(EMBEDDING_MODEL)
db = lancedb.connect(LANCEDB_PATH)
table = db.open_table(LANCEDB_TABLE)
agent = load_agent(AGENTS_DIR / "01-subtopic-writer.md")
print(f"LanceDB: {table.count_rows()} docs | Agent: {agent.name}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LanceDB: 6260 docs | Agent: Subtopic Writer


In [3]:
def parse_stub(path: Path) -> dict:
    """Parse a cluster stub file. Returns dict with keys: path, subtopics, pending, status, service, location."""
    text = path.read_text(encoding="utf-8").replace('\r\n', '\n')

    meta_match = re.search(r'<!--\s*CLUSTER_META([\s\S]*?)-->', text)
    meta = {}
    subtopics = []
    if meta_match:
        in_sub = False
        for line in meta_match.group(1).split('\n'):
            s = line.strip()
            if s.startswith('subtopics:'):
                in_sub = True; continue
            if in_sub:
                if s.startswith('- '): subtopics.append(s[2:].strip())
                elif s and not s.startswith('-'): in_sub = False
            if ':' in s and not in_sub:
                k, v = s.split(':', 1)
                meta[k.strip()] = v.strip()

    pending = [t for t in subtopics if f"## {t}\n*Content to be generated.*" in text]
    return {
        "path": path,
        "subtopics": subtopics,
        "pending": pending,
        "status": meta.get("status", "stub"),
        "service": meta.get("service", "crawlspace rot repair"),
        "location": meta.get("location", ""),
    }

all_stubs = [parse_stub(p) for p in sorted(CRAWLSPACE_ROT_DIR.glob("service_page_cluster_*.md"))]
to_process = [s for s in all_stubs if s["pending"]]
skipped = [s for s in all_stubs if not s["pending"]]

print(f"Found {len(all_stubs)} cluster files")
print(f"  To process : {len(to_process)}")
print(f"  Skip (done): {len(skipped)}")
print()
for s in to_process:
    print(f"  PENDING  {s['path'].name}  ({len(s['pending'])} subtopics)")
for s in skipped:
    print(f"  skip     {s['path'].name}")

Found 12 cluster files
  To process : 12
  Skip (done): 0

  PENDING  service_page_cluster_crawlspace-access-entry_portland.md  (1 subtopics)
  PENDING  service_page_cluster_crawlspace-access-entry_seattle.md  (1 subtopics)
  PENDING  service_page_cluster_crawlspace-floor-system-framing-repair_portland.md  (8 subtopics)
  PENDING  service_page_cluster_crawlspace-floor-system-framing-repair_seattle.md  (8 subtopics)
  PENDING  service_page_cluster_crawlspace-moisture-vapor-waterproofing_portland.md  (9 subtopics)
  PENDING  service_page_cluster_crawlspace-moisture-vapor-waterproofing_seattle.md  (9 subtopics)
  PENDING  service_page_cluster_crawlspace-structural-rot-dry-rot-repair_portland.md  (6 subtopics)
  PENDING  service_page_cluster_crawlspace-structural-rot-dry-rot-repair_seattle.md  (6 subtopics)
  PENDING  service_page_cluster_foundation-transition-ground-level-repairs_portland.md  (4 subtopics)
  PENDING  service_page_cluster_foundation-transition-ground-level-repairs_seattle.

In [4]:
def generate_subtopic(subtopic: str, doc_text: str, service: str, location_full: str) -> tuple:
    """
    Returns (content_body, ref_rows).
    content_body uses <sup>1</sup>/<sup>2</sup>; ref_rows are pipe-delimited strings.
    """
    query = f"{subtopic} {service} {location_full}"
    results = table.search(embedder.encode([query])[0]).limit(TOP_K).to_pandas()
    rag_context = "\n\n---\n\n".join(
        f"Source: {row.get('source','?')} (Page {row.get('page','N/A')})\n{row['text']}"
        for _, row in results.iterrows()
    )

    user_prompt = f"""## Full document context (understand page scope before writing):

{doc_text}

---

## Construction reference material (RAG):

{rag_context}

---

Write a ~{WORD_TARGET}-word section body for:

Subtopic: {subtopic}
Parent page: {service} — {location_full}

Rules:
- Do NOT include the section heading
- Use <sup>1</sup> and <sup>2</sup> for inline citations
- After the section body, output exactly one line: <!-- REFS -->
- Then output exactly 2 pipe-delimited reference rows (no header row), format:
  | N | Source Title | p. X | Brief note |
- Return nothing else"""

    response = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": agent.system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    raw = response.choices[0].message.content.strip()
    parts = raw.split("<!-- REFS -->", 1)
    content_body = parts[0].strip()
    ref_rows = []
    if len(parts) == 2:
        ref_rows = [l.strip() for l in parts[1].strip().splitlines() if l.strip().startswith("|")]
    return content_body, ref_rows


def write_stub(stub: dict, results_store: dict) -> None:
    """Apply generated content and references back to the file."""
    updated = stub["path"].read_text(encoding="utf-8").replace('\r\n', '\n')
    all_ref_rows = []
    global_offset = 0

    for subtopic, (content, refs) in results_store.items():
        # Renumber inline superscripts
        numbered = re.sub(r'<sup>1</sup>', f'<sup>{global_offset + 1}</sup>', content)
        numbered = re.sub(r'<sup>2</sup>', f'<sup>{global_offset + 2}</sup>', numbered)

        # Renumber ref rows
        numbered_refs = []
        for row in refs:
            row = re.sub(r'^\|\s*\d+\s*\|', f'| {global_offset + len(numbered_refs) + 1} |', row)
            numbered_refs.append(row)
        all_ref_rows.extend(numbered_refs)
        global_offset += len(refs)

        placeholder = f"## {subtopic}\n*Content to be generated.*"
        replacement = f"## {subtopic}\n{numbered}"
        updated = updated.replace(placeholder, replacement)

    if all_ref_rows:
        ref_table = (
            "\n## References\n\n"
            "| # | Source | Page | Note |\n"
            "|---|--------|------|------|\n"
            + "\n".join(all_ref_rows) + "\n"
        )
        if "## Page Metadata" in updated:
            updated = updated.replace("## Page Metadata", ref_table + "\n## Page Metadata")
        else:
            updated += ref_table

    updated = re.sub(r'(status:\s*)stub', r'\1draft', updated)
    stub["path"].write_text(updated, encoding="utf-8")

In [5]:
summary = []  # [(filename, subtopics_written, refs_written)]

for file_i, stub in enumerate(to_process):
    fname = stub["path"].name
    loc = stub["location"]
    location_full = "Portland, Oregon" if loc == "portland" else "Seattle, Washington"

    print(f"\n[{file_i+1}/{len(to_process)}] {fname}")

    # Re-read fresh before each file so doc_text is current
    doc_text = stub["path"].read_text(encoding="utf-8")
    results_store = {}

    for sub_i, subtopic in enumerate(stub["pending"]):
        print(f"  [{sub_i+1}/{len(stub['pending'])}] {subtopic} ...", end=" ", flush=True)
        content, refs = generate_subtopic(subtopic, doc_text, stub["service"], location_full)
        results_store[subtopic] = (content, refs)
        print(f"{len(content.split())} words, {len(refs)} refs")

    total_refs = sum(len(refs) for _, refs in results_store.values())

    if DRY_RUN:
        print(f"  DRY RUN — skipping write ({len(results_store)} sections, {total_refs} refs)")
    else:
        write_stub(stub, results_store)
        print(f"  Written: {len(results_store)} sections, {total_refs} refs")

    summary.append((fname, len(results_store), total_refs))

print("\n" + "="*60)
print("BATCH COMPLETE")
print("="*60)
for fname, secs, refs in summary:
    print(f"  {fname}  →  {secs} sections, {refs} refs")


[1/12] service_page_cluster_crawlspace-access-entry_portland.md
  [1/1] Crawlspace Access Door Installation ... 158 words, 2 refs
  Written: 1 sections, 2 refs

[2/12] service_page_cluster_crawlspace-access-entry_seattle.md
  [1/1] Crawlspace Access Door Installation ... 174 words, 2 refs
  Written: 1 sections, 2 refs

[3/12] service_page_cluster_crawlspace-floor-system-framing-repair_portland.md
  [1/8] Crawlspace Floor Joist Repair ... 160 words, 2 refs
  [2/8] Crawlspace Rim Joist Repair ... 154 words, 2 refs
  [3/8] Crawlspace Sill Plate Repair ... 174 words, 2 refs
  [4/8] Crawlspace Foundation Framing Repair ... 170 words, 2 refs
  [5/8] Crawlspace Structural Reinforcement ... 170 words, 2 refs
  [6/8] Crawlspace Joist Sistering ... 183 words, 2 refs
  [7/8] Crawlspace Beam Reinforcement ... 169 words, 2 refs
  [8/8] Crawlspace Structural Framing ... 152 words, 2 refs
  Written: 8 sections, 16 refs

[4/12] service_page_cluster_crawlspace-floor-system-framing-repair_seattle.md
  